In [ ]:
import getpass
import hashlib
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import requests
from requests.structures import CaseInsensitiveDict

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)
FIXTURES = Path("fixtures")

### The offline switch


In [ ]:
OFFLINE = os.environ.get("OFFLINE") == "1"
POLITE_DELAY = 0.0 if OFFLINE else 1.0   # seconds between live requests


def cache_key(url, params=None):
    """A short, stable name for one (url, params) pair.
    """
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


class FixtureResponse:

    def __init__(self, record):
        self.url = record.get("final_url", record["url"])
        self.status_code = record["status_code"]
        self.headers = CaseInsensitiveDict(record["headers"])
        self._body = record["body"]

    def json(self):
        return self._body

    @property
    def links(self):
        header = self.headers.get("link")
        if not header:
            return {}
        return {l["rel"]: l for l in requests.utils.parse_header_links(header)}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"{self.status_code} for {self.url}")


def offline_get(url, params=None, **kwargs):
    path = FIXTURES / f"{cache_key(url, params)}.json"
    if not path.exists():
        raise RuntimeError(
            f"no fixture for {url} {params}")
    return FixtureResponse(json.loads(path.read_text()))

if OFFLINE:
    requests.get = offline_get
    print("OFFLINE:", len(list(FIXTURES.glob("*.json"))), "fixtures loaded")
else:
    print("live requests")

live requests
